# Registration: moving segmentations between views

Adapted from the RSNA 2024 lumbar spine registration notebook (`registration.ipynb`) to knee MRI. The goal is an **axial segmentation** without running a model on axial images: nnU-Net (`02_segmentation_nnunet.ipynb`) was trained on 3D sagittal knees and is unreliable on axial series, so its sagittal and coronal predictions are resampled onto the axial voxel grid instead.

What changes from the lumbar notebook:

- **Full 3D geometry.** The lumbar notebook maps planes through one coordinate of `ImagePositionPatient` (e.g. axial z = sagittal image y), which assumes axis-aligned planes. Knee series are routinely oblique (here ~5-8 degrees), so each series gets its full geometry (origin, direction cosines from `ImageOrientationPatient`, spacing) and points are mapped between voxel grids in world space (LPS, mm). Helpers live in `src/knee_registration.py`.
- **Frame of reference.** Every series in this dataset has its own `FrameOfReferenceUID` (most likely remapped by the anonymisation), so the tags cannot confirm that series share coordinates. Alignment is therefore checked on the images, and an optional **rigid registration** (mutual information, since contrasts differ) corrects patient motion between acquisitions.
- **Thick slices.** A sagittal series samples left-right every 3-4 mm, which becomes an *in-plane* axis of an axial slice. Labels are resampled as one-hot probabilities with linear interpolation, and sagittal and coronal (fine left-right sampling) are fused by a weighted soft vote.
- Uses the nnU-Net predictions cached by `02_segmentation_nnunet.ipynb` for the same automatically chosen study. Run that notebook first for another study.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import knee_registration as kr

OUT = ROOT / 'output'
NN = OUT / 'segmentation' / 'nnunet'
SAVE = OUT / 'segmentation' / 'registered'
STUDY_UID = None        # None: same automatic choice as 02 / 03 (labeled study with fluid-sensitive series in all 3 views)
TARGET_SLICES = 30
REFINE = True           # rigid registration on top of the DICOM geometry
WEIGHTS = {'Sagittal': 1.0, 'Coronal': 0.5}   # soft-vote weights of the source views (nnU-Net is best on sagittal)
SAVE.mkdir(parents=True, exist_ok=True)

LABELS = {1: 'patellar cartilage', 2: 'femoral cartilage', 3: 'medial tibial cartilage', 4: 'lateral tibial cartilage',
          5: 'medial meniscus', 6: 'lateral meniscus', 7: 'femur', 8: 'tibia', 9: 'patella'}
COLORS = ['#ffd92f', '#e7298a', '#1b9e77', '#66c2a5', '#fc8d62', '#8da0cb', '#e5c494', '#a6d854', '#b3b3b3']
CMAP = ListedColormap(['none'] + COLORS)
LEGEND = [Patch(color=COLORS[k - 1], label=v) for k, v in LABELS.items()]

## 1. Series and their geometry

In [ ]:
idx = pd.read_csv(OUT / 'series_index.csv')
train = pd.read_csv(ROOT / 'input' / 'train.csv', usecols=['StudyInstanceUID', 'ACL'])
fluid = idx[idx.Fluid_Sensitive == 1]
complete = fluid.groupby('StudyInstanceUID').Anatomical_Plane.nunique()
if STUDY_UID is None:
    STUDY_UID = sorted(set(complete[complete == 3].index) & set(train.dropna().StudyInstanceUID))[0]
cand = fluid[fluid.StudyInstanceUID == STUDY_UID]
picked = (cand.assign(d=(cand.n_slices - TARGET_SLICES).abs()).sort_values('d').groupby('Anatomical_Plane').head(1)
              .set_index('Anatomical_Plane').loc[['Sagittal', 'Coronal', 'Axial']])

views = {}
rows = []
for view, r in picked.iterrows():
    img = kr.series_image(r.SeriesInstanceUID)
    g = kr.series_geometry(r.SeriesInstanceUID)
    normal = np.array(g['direction']).reshape(3, 3)[:, 2]
    axis = {'Sagittal': 0, 'Coronal': 1, 'Axial': 2}[view]
    views[view] = dict(uid=r.SeriesInstanceUID, img=img, vol=sitk.GetArrayFromImage(img),
                       case=f'{view.lower()}_{r.SeriesInstanceUID[-8:]}')
    rows.append(dict(view=view, series=r.SeriesInstanceUID[-8:], size_xyz=img.GetSize(),
                     spacing_mm=tuple(round(s, 3) for s in img.GetSpacing()),
                     normal_LPS=tuple(np.round(normal, 3)),
                     obliquity_deg=round(float(np.degrees(np.arccos(min(1, abs(normal[axis]))))), 1),
                     uniform_spacing=g['uniform']))
print('study', STUDY_UID)
pd.DataFrame(rows).set_index('view')

`obliquity_deg` is the angle between the slice normal and the patient axis the plane is named after. Anything above a degree or two breaks the lumbar notebook's one-coordinate mapping.

## 2. nnU-Net segmentations, per view

The cascade predictions from `02_segmentation_nnunet.ipynb`, on their own series. The geometry rebuilt above is checked against the NIfTI headers nnU-Net wrote.

In [ ]:
for view, v in views.items():
    p = NN / v['case'] / 'cascade' / f"{v['case']}.nii.gz"
    assert p.exists(), f'{p} missing: run 02_segmentation_nnunet.ipynb on this study first'
    v['seg_img'] = sitk.ReadImage(str(p))
    v['seg'] = sitk.GetArrayFromImage(v['seg_img']).astype(np.uint8)
    same = (np.allclose(v['seg_img'].GetOrigin(), v['img'].GetOrigin(), atol=1e-3)
            and np.allclose(v['seg_img'].GetDirection(), v['img'].GetDirection(), atol=1e-4)
            and np.allclose(v['seg_img'].GetSpacing(), v['img'].GetSpacing(), atol=1e-4))
    print(f'{view:9s} seg {v["seg"].shape}  geometry matches the series: {same}')


def norm(img):
    lo, hi = np.percentile(img, [1, 99.5])
    return np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)


def show_seg(ax_row, vol, seg, title, n=5):
    nz = np.flatnonzero(seg.any((1, 2)))
    zs = np.linspace(nz[0], nz[-1], n).round().astype(int) if len(nz) else np.linspace(0, len(vol) - 1, n).round().astype(int)
    for a, z in zip(ax_row, zs):
        a.imshow(norm(vol[z]), cmap='gray')
        a.imshow(np.ma.masked_equal(seg[z], 0), cmap=CMAP, vmin=0, vmax=9, alpha=0.5, interpolation='nearest')
        a.set_title(f'{title} slice {z}/{len(vol) - 1}', fontsize=9); a.axis('off')


fig, ax = plt.subplots(3, 5, figsize=(20, 12.5))
for row, (view, v) in zip(ax, views.items()):
    show_seg(row, v['vol'], v['seg'], view)
fig.legend(handles=LEGEND, loc='lower center', ncol=9, fontsize=9)
plt.suptitle('nnU-Net cascade on each view', y=0.995); plt.tight_layout(rect=(0, 0.03, 1, 1)); plt.show()

vox = {view: {LABELS[k]: int((v['seg'] == k).sum()) * np.prod(v['img'].GetSpacing()) / 1000 for k in LABELS}
       for view, v in views.items()}
pd.DataFrame(vox).round(2).rename_axis('volume (ml)')

The volumes are for the same knee, so large differences between views (beyond field of view) point at the views nnU-Net handles badly.

## 3. Cross-references between views

The lumbar notebook's "axial levels drawn on the sagittal", done with the full geometry: each line is the intersection of another series' slice plane with the displayed slice (`kr.slice_lines`), so oblique stacks give tilted, non-parallel lines where they should.

In [ ]:
pairs = [('Sagittal', 'Axial'), ('Coronal', 'Axial'), ('Axial', 'Sagittal'), ('Axial', 'Coronal')]
fig, axes = plt.subplots(1, 4, figsize=(22, 5.8))
for a, (shown, other) in zip(axes, pairs):
    v = views[shown]
    k = len(v['vol']) // 2
    a.imshow(norm(v['vol'][k]), cmap='gray')
    for k2, p0, p1 in kr.slice_lines(v['img'], k, views[other]['img']):
        a.plot([p0[0], p1[0]], [p0[1], p1[1]], 'r--', lw=0.6)
    a.set_title(f'{other} slices on the middle {shown.lower()} slice', fontsize=10); a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# A few axial slices next to the sagittal and coronal middle slices, with that axial plane highlighted.
ax_v = views['Axial']
ks = np.linspace(4, len(ax_v['vol']) - 5, 4).round().astype(int)
fig, axes = plt.subplots(len(ks), 3, figsize=(14, 4.8 * len(ks)))
for row, k in zip(axes, ks):
    row[0].imshow(norm(ax_v['vol'][k]), cmap='gray'); row[0].set_title(f'axial slice {k}')
    for a, view in zip(row[1:], ['Sagittal', 'Coronal']):
        v = views[view]; kk = len(v['vol']) // 2
        a.imshow(norm(v['vol'][kk]), cmap='gray')
        for k2, p0, p1 in kr.slice_lines(v['img'], kk, ax_v['img']):
            if k2 == k:
                a.plot([p0[0], p1[0]], [p0[1], p1[1]], 'r-', lw=1.5)
        a.set_title(f'{view.lower()} slice {kk}: axial {k} in red')
    for a in row: a.axis('off')
plt.tight_layout(); plt.show()

## 4. Are the series aligned?

The sagittal and coronal images are resampled onto the axial grid and shown as a checkerboard with the axial image: anatomy (bone edges, joint line, patella) should continue across tile borders. Before = DICOM geometry only; after = plus rigid registration (`kr.rigid_refine`, Mattes mutual information, lower is better). Black tiles are outside the source series' field of view.

In [ ]:
fixed = ax_v['img']
for view in WEIGHTS:
    v = views[view]
    tx, before, after = kr.rigid_refine(fixed, v['img'])
    rot, shift = kr.transform_summary(tx)
    v['tx'] = tx if REFINE else None
    print(f'{view:9s} MI {before:.4f} -> {after:.4f}   correction: rotation {rot:.2f} deg, translation {shift:.2f} mm')


def checkerboard(a, b, n=8):
    yy, xx = np.indices(a.shape)
    m = ((yy * n // a.shape[0]) + (xx * n // a.shape[1])) % 2 == 0
    return np.where(m, a, b)


ks = np.linspace(6, len(ax_v['vol']) - 7, 3).round().astype(int)
fig, axes = plt.subplots(len(WEIGHTS) * 2, len(ks), figsize=(5.2 * len(ks), 5.4 * 2 * len(WEIGHTS)))
r = 0
for view in WEIGHTS:
    v = views[view]
    for label, tx in [('DICOM geometry', None), ('+ rigid registration', kr.rigid_refine(fixed, v['img'])[0])]:
        moved = sitk.GetArrayFromImage(kr.resample(v['img'], fixed, tx))
        for a, k in zip(axes[r], ks):
            a.imshow(checkerboard(norm(ax_v['vol'][k]), norm(moved[k])), cmap='gray')
            a.set_title(f'axial {k} x {view.lower()}: {label}', fontsize=9); a.axis('off')
        r += 1
plt.tight_layout(); plt.show()

## 5. Segmentation transferred to the axial series

Each source segmentation becomes one-hot probabilities on the axial grid (`kr.label_probabilities`, linear interpolation), then a weighted soft vote (`kr.fuse`, `WEIGHTS`) where each view only votes inside its own field of view. The native nnU-Net axial prediction is shown for comparison, it is not part of the vote.

In [ ]:
probs, covs, single = {}, {}, {}
for view in WEIGHTS:
    v = views[view]
    probs[view], covs[view] = kr.label_probabilities(v['seg_img'], fixed, len(LABELS), v.get('tx'))
    single[view] = kr.fuse([probs[view]], [covs[view]])
fused = kr.fuse([probs[v] for v in WEIGHTS], [covs[v] for v in WEIGHTS], [WEIGHTS[v] for v in WEIGHTS])
print('axial voxels covered by: ' + ', '.join(f'{v} {(covs[v] >= 0.5).mean():.0%}' for v in WEIGHTS))

columns = [('nnU-Net on axial', ax_v['seg'])] + [(f'from {v.lower()}', single[v]) for v in WEIGHTS] + [('fused', fused)]
nz = np.flatnonzero(fused.any((1, 2)))
ks = np.linspace(nz[0], nz[-1], 6).round().astype(int)
fig, axes = plt.subplots(len(ks), len(columns) + 1, figsize=(4.2 * (len(columns) + 1), 4.3 * len(ks)))
for row, k in zip(axes, ks):
    img = norm(ax_v['vol'][k])
    row[0].imshow(img, cmap='gray'); row[0].set_title(f'axial slice {k}', fontsize=10)
    for a, (name, seg) in zip(row[1:], columns):
        a.imshow(img, cmap='gray')
        a.imshow(np.ma.masked_equal(seg[k], 0), cmap=CMAP, vmin=0, vmax=9, alpha=0.5, interpolation='nearest')
        a.set_title(name, fontsize=10)
    for a in row: a.axis('off')
fig.legend(handles=LEGEND, loc='lower center', ncol=9, fontsize=10)
plt.tight_layout(rect=(0, 0.02, 1, 1)); plt.show()

### Agreement between sources

Dice per structure on the axial grid, restricted to voxels both sources see. There is no ground truth: high agreement between sagittal and coronal (two independent acquisitions and predictions) is evidence that both are right; the native axial column shows how far nnU-Net on axial images is from them.

In [ ]:
both = (covs['Sagittal'] >= 0.5) & (covs['Coronal'] >= 0.5)
comparisons = {'sagittal vs coronal': (single['Sagittal'], single['Coronal']),
               'axial nnU-Net vs fused': (ax_v['seg'], fused),
               'axial nnU-Net vs sagittal': (ax_v['seg'], single['Sagittal'])}
table = {name: {LABELS[k]: kr.dice((a == k) & both, (b == k) & both) for k in LABELS}
         for name, (a, b) in comparisons.items()}
pd.DataFrame(table).round(2)

## 6. Save

In [ ]:
out = SAVE / f"{ax_v['case']}_fused.nii.gz"
seg_img = kr.with_geometry(fused, fixed)
sitk.WriteImage(seg_img, str(out))
print('wrote', out, fused.shape, 'labels', sorted(int(v) for v in np.unique(fused) if v))

### How to read this

- **Section 4 decides whether the rest is usable.** If structures jump across checkerboard tiles even after registration, the knee moved between series in a non-rigid way (e.g. flexion change), and transferred labels will be off by that much.
- **Resolution.** The transferred labels inherit the source's resolution along its slice axis: sagittal gives left-right steps of ~3 mm, coronal anterior-posterior steps of ~3.5 mm. Fusing the two is what makes axial in-plane shapes reasonable; thin structures (cartilage, menisci) remain the least reliable.
- **Coverage.** Only the part of the axial stack inside the sagittal / coronal fields of view gets labels: axial series often extend further up the thigh and down the shin.
- **Towards all studies.** Everything here is in `src/knee_registration.py`; running nnU-Net on the sagittal (and coronal) fluid-sensitive series of each study and calling `label_probabilities` + `fuse` gives an axial segmentation per study.